In [ ]:
from cryoswath.misc import load_glacier_outlines, fill_missing_coords
from dask.distributed import Client, LocalCluster
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from cryoswath_paper_workflow.workflow_support import (
    calculate_mean_elevation_and_uncertainty,
    get_interp_unc,
    get_interp_unc_corr,
    fit_deseasoned_trend,
    to_yaml,
)
from typing import Protocol
import xarray as xr


class snakemake_mock(Protocol):
    input: dict[int, str]
    output: dict[int, str]
    wildcards: str
    threads: int


snakemake = globals().get("snakemake", snakemake_mock)

In [ ]:
if snakemake.wildcards.token == "penny":
    complex_id = ["RGI2000-v7.0-C-04-05230"]
elif snakemake.wildcards.token == "barnes":
    complex_id = ["RGI2000-v7.0-C-04-00362"]
elif snakemake.wildcards.token == "sukkerqara":
    complex_id = ["RGI2000-v7.0-C-05-08741", "RGI2000-v7.0-C-05-08918", "RGI2000-v7.0-C-05-09193"]
elif snakemake.wildcards.token == "scandinavian":
    complex_id = [
        "RGI2000-v7.0-C-08-01742",  # Hardangerjoekulen
        "RGI2000-v7.0-C-08-01770",  # Jostedalsbreen
        "RGI2000-v7.0-C-08-02204",  # Soendre Folgefonna
        "RGI2000-v7.0-C-08-02200",  # Nordre Folgefonna
        "RGI2000-v7.0-C-08-00291",  # Svartisen-Vestisen
        "RGI2000-v7.0-C-08-00294",  # Svartisen-Oestisen
        "RGI2000-v7.0-C-08-00385",  # Blaemanissen
    ]
elif snakemake.wildcards.token == "chitina":
    complex_id = ["RGI2000-v7.0-G-01-16307"]
else:
    raise Exception(f"Not implemented for {snakemake.wildcards.token}")
icecap_gdf = load_glacier_outlines(complex_id, union=False)

In [ ]:
df_list = []
for file in snakemake.input:
    if not file.endswith("monthly_500x500m.nc"):
        continue
    with xr.open_dataset(
        file, decode_coords="all", engine="h5netcdf"
    ).dropna("time", how="all") as ds:
        print(file, ds.rio.crs)
        # ds.elev_diff.mean("time").plot()
        # plt.show()
        ds = (
            ds.drop_vars(
                [
                    "elev_diff_obs_count",
                    "elev_diff_ref",
                    "basin_id",
                    "group_id",
                ]
            )
            .transpose("time", "y", "x")
            .rio.clip(icecap_gdf.to_crs(ds.rio.crs).make_valid())
        ).map(lambda da: da.astype("f4"))
        ds.elev_diff.mean("time").plot()
        plt.show()
        crs = globals().get("crs", ds.rio.crs)
        print(crs.__class__, crs)
        df_list.append(
            ds.rio.reproject(crs)
            .to_dataframe()
            .drop(columns="spatial_ref")
            .dropna(how="any")
        )
tmp = pd.concat(df_list, axis=0)
ds = (
    tmp[~tmp.index.duplicated()]
    .to_xarray()
    .rio.write_crs(crs)
    .sortby(["time", "x", "y"])
)
if snakemake.wildcards.token == "scandinavian":
    ds = fill_missing_coords(ds)
ds

In [ ]:
# check projection
ds.elev_diff.mean("time").T.plot(robust=True, cmap="RdYlBu")
icecap_gdf.boundary.to_crs(crs).plot(ax=plt.gca())  # pyright: ignore[reportAttributeAccessIssue]

In [ ]:
with (
    LocalCluster(n_workers=snakemake.threads, dashboard_address=None) as cluster,
    Client(cluster),
):
    if snakemake.wildcards.token == "scandinavian":
        # treat separately
        icecap_gdf["glac_name"] = [
            "Hardangerjoekulen",
            "Jostedalsbreen",
            "Soendre Folgefonna",
            "Nordre Folgefonna",
            "Svartisen-Vestisen",
            "Svartisen-Oestisen",
            "Blaemanissen",
        ]
        results = {row.glac_name: calculate_mean_elevation_and_uncertainty(
            ds.rio.clip([row.geometry]),
            get_interp_unc(snakemake.input["unc_tbl"]),
            get_interp_unc_corr(snakemake.input["corr_tbl"]),
        ).to_dataframe() for _, row in icecap_gdf.to_crs(ds.rio.crs).iterrows()}
    else:
        results = calculate_mean_elevation_and_uncertainty(
            ds,
            get_interp_unc(snakemake.input["unc_tbl"]),
            get_interp_unc_corr(snakemake.input["corr_tbl"]),
        ).to_dataframe()
print(results)

In [ ]:
def quick_plot(df, title=None):
    plt.fill_between(
        df.index,
        *[df.elevation + x * df.uncertainty for x in [-1, 1]],
        color="tab:gray",
    )
    plt.plot(df.index, df.elevation)
    plt.title(title or "Avg. surface elevation")
    plt.show()


if isinstance(results, pd.DataFrame):
    quick_plot(results)
else:
    for l, df in results.items():
        quick_plot(df, l)

In [ ]:
def fit(df, area_km2):
    fit_res = fit_deseasoned_trend(
        df.elevation.resample("QS-AUG", label="left").first().loc["2010-08":]
    )
    trend = fit_res.params.trend * 4
    trend_unc = 2 * fit_res.bse.trend * 4
    density = 0.85  # tonnes per cubicmeter
    dens_unc = 0.06
    area = area_km2 * 1e6  # pyright: ignore[reportAttributeAccessIssue]
    area_unc = 0.05
    mass_trend = trend * area * density / 1e9  # Gt/yr
    mass_unc = (
        np.abs(mass_trend)
        * (np.array([trend_unc / trend, dens_unc, area_unc]) ** 2).sum() ** 0.5
    )
    print(f"Surface trend: {trend:.3f}", f"+- {trend_unc:.3f}", "m/yr")
    print(f"Mass trend: {mass_trend:.2f}", f"+- {mass_unc:.2f}", "Gt/yr")
    print("(uncertainty from fit stats and density)")
    print("Student t =", fit_res.tvalues.trend)
    print("p =", fit_res.pvalues.trend)
    sm.graphics.plot_fit(fit_res, 1)
    plt.show()
    print(fit_res.summary())
    return {
        k: float(v)
        for k, v in [
            ("specific", trend),
            ("sp_err", trend_unc),
            ("mass", mass_trend),
            ("m_err", mass_unc),
            ("p-val", fit_res.pvalues.trend),
        ]
    }


if isinstance(results, pd.DataFrame):
    to_yaml(fit(results, icecap_gdf.area_km2.sum()),
        snakemake.output[0],
    )
else:
    out_dict = dict()
    for l, df in results.items():
        out_dict[l] = fit(df, icecap_gdf[icecap_gdf.glac_name == l].area_km2.iloc[0])
    to_yaml(out_dict,
        snakemake.output[0],
    )